# Publication copy
The code and analytical commentary are preserved. Stored execution outputs were cleared to avoid publishing respondent previews or local execution details. Vetted aggregate tables and figures are provided in `course_artifacts`; see the repository results guide. The fully executed originals remain in the local course workspace. This notebook needs the separately acquired survey data and protected fit caches for a complete replay. No models were refitted for publication.


# Other clustering algorithms on the complete response data

This variant extends v3, preserving the complete-response population, the separate personal and known-person harm indicators, within-person relative worry and reduced-weight overall worry. We test GMM, Ward and average-linkage agglomerative clustering, and DBSCAN. K-means B and E remain saved reference comparators, not assumed winners.

We retain encodings 0–1–2 and 0–1–3, all three worry/harm balances and k=2–10 where k is specified. Encoding 0–1–5 remains in the earlier sensitivity results, not this search. No imputation, PCA, alternative scaling or geographic clustering is introduced. Held-out-country respondents remain unused.

**Read before running:** the reference search and the 50+50 repetition stage are separately enabled below. Both are off initially because they are substantial computations. Run the preparation and smoke-test cells first. A smoke test confirms code paths, not scientific results. Run the full reference search, review fitting diagnostics, then enable the complete stability stage. No reduced-repetition run can qualify a winner under our rules. This notebook is self-contained; caches and tables are its outputs, not additional source files.

## 1. Declare the search and execution controls

**What and why:** fix the representation, model parameters, random seeds, repetitions and memory guards before examining algorithm results. The two execution switches prevent an accidental long run. They are not part of the scientific fingerprint. Model fits remain one numerical thread each; hierarchy and radius graphs run sequentially to limit peak memory. Failed or memory-guarded fits are recorded, not silently replaced by a smaller sample.

There are 108 GMM candidates (two covariance forms), 108 hierarchical candidates (two linkages) and up to 72 DBSCAN parameter combinations before removing repeated radii. GMM regularization is fixed across candidates in scaled squared units; a covariance at the floor is a diagnostic warning, not proof of a distinct type.

In [ ]:
from pathlib import Path
import json, hashlib, itertools
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from scipy.optimize import linear_sum_assignment
from joblib import Parallel, delayed, parallel_config
from threadpoolctl import threadpool_limits

ROOT=Path.cwd()
if not (ROOT/'course_artifacts').exists(): ROOT=ROOT.parent
INPUT=ROOT/'course_artifacts/week04_search_v4/stage1_eligible_development.parquet'
OUTPUT=ROOT/'course_artifacts/week04_other_algorithms_v1'
OUTPUT.mkdir(exist_ok=True)
(OUTPUT/'fits').mkdir(exist_ok=True)
WORRY=['worry_'+d for d in ['food','water','crime','severe_weather','traffic','mental_health']]
DOMAINS=[c.removeprefix('worry_') for c in WORRY]+['work']
PERSONAL=['harm_'+d+'_personal' for d in DOMAINS]
KNOWN=['harm_'+d+'_known_person' for d in DOMAINS]
FEATURES=WORRY+PERSONAL+KNOWN
BALANCES={'balanced':.50,'worry_emphasized':.75,'harm_emphasized':.25}
K_VALUES=range(2,11)
REPETITIONS=50
SEED=20261005
N_JOBS=4
ENCODINGS={'012':2.,'013':3.}
RUN_REFERENCE=False
RUN_STABILITY=False
GMM_COVARIANCES=['diag','full']
GMM_N_INIT=5
GMM_MAX_ITER=300
GMM_REG_COVAR=1e-3
AGG_PER_COUNTRY=150
AGG_MAX_DISTANCE_BYTES=2_000_000_000
DBSCAN_MIN_SAMPLES=[20,50,100]
DBSCAN_RADIUS_QUANTILES=[.50,.70,.85,.95]
DBSCAN_PROBES=2000
DBSCAN_MAX_EDGES=5_000_000
DBSCAN_CHUNK_SIZE=250
DESIGN={'encodings':ENCODINGS,'balances':BALANCES,'k':list(K_VALUES),'repetitions':REPETITIONS,
        'seed':SEED,'fraction':.8,'imputation':'none','overall_worry':'reduced_same_multiplier',
        'gmm_covariances':GMM_COVARIANCES,'gmm_n_init':GMM_N_INIT,'gmm_max_iter':GMM_MAX_ITER,
        'gmm_reg_covar':GMM_REG_COVAR,'gmm_draw':'equal_country_bootstrap_about_training_N',
        'agg_per_country':AGG_PER_COUNTRY,'agg_linkages':['ward','average'],
        'agg_transport':'nearest_native_cluster_centroid','agg_max_distance_bytes':AGG_MAX_DISTANCE_BYTES,
        'dbscan_min_samples':DBSCAN_MIN_SAMPLES,'dbscan_quantiles':DBSCAN_RADIUS_QUANTILES,
        'dbscan_probes':DBSCAN_PROBES,'dbscan_edges':DBSCAN_MAX_EDGES,'dbscan_chunk':DBSCAN_CHUNK_SIZE,
        'dbscan_density_mass':'equal_country_weights_times_complete_reference_N',
        'dbscan_transport':'nearest_core_within_eps_preserve_native_training_labels',
        'strata':['GlobalRegion','CountryIncomeLevel'],'version':1}
FINGERPRINT=hashlib.sha256(INPUT.read_bytes()+json.dumps(DESIGN,sort_keys=True).encode()).hexdigest()
(OUTPUT/'design.json').write_text(json.dumps(DESIGN,indent=2))
(OUTPUT/'trees').mkdir(exist_ok=True)
sns.set_theme(style='whitegrid')
def show(title,table):
    display(Markdown('### '+title));display(table)
def country_weights(frame):
    return (frame.WGT/frame.groupby('COUNTRY_ISO3').WGT.transform('sum')/frame.COUNTRY_ISO3.nunique()).to_numpy()

import sklearn,scipy
print("Installed versions:",sklearn.__version__,scipy.__version__,np.__version__,pd.__version__)


## 2. Restrict to complete responses and audit exclusions

**What this cell does:** first identifies the previous eligible cohort, then retains only complete worry and harm responses. It records losses overall and by country, checks identifiers and valid values, and recalculates equal-country weights.

**Why:** the new clustering does not depend on imputed answers. Complete cases may nevertheless be selected systematically; keeping all countries does not remove selection bias within countries. Percentages excluded under old weights describe the original population; newly calculated weights describe the retained one.

In [ ]:
raw=pd.read_parquet(INPUT)
assert raw.country_split.eq('train').all()
eligible=raw[WORRY].notna().sum(axis=1).ge(4)&raw[PERSONAL].notna().sum(axis=1).ge(4)
previous=raw.loc[eligible].copy()
complete=previous[FEATURES].notna().all(axis=1)
previous_weights=country_weights(previous)
data=previous.loc[complete].copy().reset_index(drop=True)
assert np.isfinite(data[FEATURES].to_numpy(float)).all()
assert data[WORRY].isin([0,1,2]).all().all()
assert data[PERSONAL+KNOWN].isin([0,1]).all().all()
audit=pd.DataFrame({'previous_eligible':[len(previous)],'retained':[len(data)],'excluded':[int((~complete).sum())],
                    'excluded_pct':[100*(~complete).mean()],
                    'excluded_previous_weight_pct':[100*previous_weights[~complete.to_numpy()].sum()]})
audit.to_csv(OUTPUT/'complete_case_audit.csv',index=False)
country_audit=previous.assign(complete=complete).groupby('COUNTRY_ISO3').complete.agg(['size','sum','mean'])
country_audit.columns=['previous_eligible','retained','retained_fraction']
country_audit.to_csv(OUTPUT/'complete_case_country_audit.csv')
show('Complete-response retention and exclusions',audit)
show('Country-level complete-response retention',country_audit)
assert data.WPID_RANDOM.is_unique
assert np.isfinite(data.WGT).all() and data.WGT.gt(0).all()
weights=country_weights(data)
countries=data.COUNTRY_ISO3.to_numpy()
country_rows={c:np.flatnonzero(countries==c) for c in sorted(set(countries))}
registry=pd.read_csv(ROOT/'course_artifacts/week03_country_split.csv')
registry=registry.loc[registry.country_split.eq('train')&registry.COUNTRY_ISO3.isin(country_rows)].copy()
assert len(registry)==len(country_rows) and registry.COUNTRY_ISO3.is_unique
show('Fixed development population',pd.DataFrame({'respondents':[len(data)],'countries':[len(country_rows)],'excluded_from_input':[len(raw)-len(data)]}))


## 3. Keep the same two perturbation designs

**What and why:** regenerate the 50 respondent and 50 region × income-stratified country samples using v3's seed, then verify exact index equality against its saved plans. Recompute equal-country weights inside every fit. Predictions use the same complete-response cohort. The country scheme always includes some small-stratum countries, limiting omission sensitivity.

In [ ]:
groups=registry.groupby(['GlobalRegion','CountryIncomeLevel'],sort=True)
quota=groups.size().rename('available').to_frame()
quota['selected']=np.maximum(1,np.floor(.8*quota.available).astype(int))
target=round(.8*len(registry))
while quota.selected.sum()<target:
    remainder=(.8*quota.available-quota.selected).where(quota.selected<quota.available,-np.inf)
    quota.loc[remainder.idxmax(),'selected']+=1
quota['always_included']=quota.selected.eq(quota.available)
quota.to_csv(OUTPUT/'stratum_quotas.csv')
show('Country retention by region × income stratum',quota)
print('Countries per repetition:',target,'; always included:',int(quota.loc[quota.always_included,'available'].sum()))
plans={'respondents':[],'countries':[]}
plan_rows=[]
for repeat in range(REPETITIONS):
    rng=np.random.default_rng(SEED+repeat)
    rows=np.sort(np.concatenate([rng.choice(idx,round(.8*len(idx)),replace=False) for idx in country_rows.values()]))
    plans['respondents'].append(rows)
    selected=[]
    for key,frame in groups:
        selected.extend(rng.choice(sorted(frame.COUNTRY_ISO3),int(quota.loc[key,'selected']),replace=False))
    rows=np.flatnonzero(np.isin(countries,selected))
    plans['countries'].append(rows)
    for kind in plans:
        idx=plans[kind][-1]
        plan_rows.append({'experiment':kind,'repeat':repeat,'respondents':len(idx),'countries':len(set(countries[idx])),'country_list':' '.join(sorted(set(countries[idx])))})
pd.DataFrame(plan_rows).to_csv(OUTPUT/'sampling_plans.csv',index=False)
joblib.dump(plans,OUTPUT/'sampling_indices.joblib')

previous_plans=joblib.load(ROOT/'course_artifacts/week04_experiment1_complete_overall_worry_v3/sampling_indices.joblib')
assert all(np.array_equal(a,b) for kind in plans for a,b in zip(plans[kind],previous_plans[kind]))


## 4. Preserve the v3 geometry

**What and why:** recode, range-normalize, centre each person's six worry values and append their mean with the same multiplier as an individual relative-worry feature. Harm remains uncentred. This restores intensity information at one-sixth its equivalent uncentred squared-distance influence. All algorithms receive the same 21-column matrix for a given specification.

The same information is represented redundantly: six centred values sum to zero. Full GMM covariance is therefore singular without regularization. Binary harm also violates Gaussian assumptions. We report convergence, ambiguity and covariance-floor behavior rather than interpret components automatically as natural populations.

In [ ]:
def encoded(frame,top):
    values=frame[FEATURES].to_numpy(float).copy()
    values[:,:6]=np.where(values[:,:6]==2,top,values[:,:6])
    return values
def transform(frame,alpha,top):
    values=encoded(frame,top)
    worry=values[:,:6]/top
    overall=worry.mean(axis=1,keepdims=True)
    relative=worry-overall
    return np.column_stack([relative*np.sqrt(alpha)/np.sqrt(6),
                            overall*np.sqrt(alpha)/np.sqrt(6),
                            values[:,6:]*np.sqrt((1-alpha)/2)/np.sqrt(7)])
for top in ENCODINGS.values():
    worry=encoded(data,top)[:,:6]/top
    overall=worry.mean(axis=1,keepdims=True)
    relative=worry-overall
    assert np.allclose(relative.mean(axis=1),0)
    assert np.allclose(relative+overall,worry)
    delta=worry[:100]-worry[100:200]
    shape=relative[:100]-relative[100:200]
    intensity=overall[:100]-overall[100:200]
    assert np.allclose((delta**2).sum(axis=1),(shape**2).sum(axis=1)+6*intensity[:,0]**2)

for encoding,top in ENCODINGS.items():
    check=transform(data,.5,top)
    assert check.shape==(len(data),21) and np.isfinite(check).all()
    assert np.allclose(check[:,:6].mean(axis=1),0)
print('Representation checks passed.')


## 5. Define the fitting population for each algorithm

**What and why:** scikit-learn GMM has no sample-weight fitting argument. Draw an equal number per training country with replacement, proportional to its survey weights; use approximately the original training N overall. This approximates the equal-country target rather than fitting unweighted original rows. Each repetition redraws, so measured variability includes this approximation.

Hierarchy also lacks respondent weights and native prediction. Use the same country-balanced weighted-bootstrap principle, with 150 draws per country (16,950 reference leaves, not 30 respondents per country). Fit an exact Ward or average-linkage tree on those draws, then assign the full cohort to nearest native-cluster centroids. This is explicitly a **hierarchy plus centroid-extension pipeline**, not full-population native hierarchy. Report native-versus-extension disagreement on the fitting draw; high disagreement warns that the extension changes the hierarchy. A dendrogram's exact nesting does not guarantee nesting after centroid extension.

DBSCAN fits every distinct training pattern with aggregated survey-weight mass. Its density weights sum to the fixed complete-cohort N in each sample, preserving density-mass units under subsampling. Its radius is calibrated on the reference development data, then frozen during perturbations. No noise label is treated as a substantive profile.

In [ ]:
from scipy.cluster.hierarchy import linkage,cut_tree
from scipy.spatial.distance import cdist
from scipy.sparse import vstack
from sklearn.mixture import GaussianMixture
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
import warnings

def country_draw(frame,per_country,seed):
    rng=np.random.default_rng(seed)
    draws=[]
    for _,group in frame.groupby('COUNTRY_ISO3',sort=True):
        probability=group.WGT.to_numpy(float);probability=probability/probability.sum()
        draws.append(rng.choice(group.index.to_numpy(),per_country,replace=True,p=probability))
    return np.concatenate(draws)

def predict_centroids(matrix,centres):
    result=np.empty(len(matrix),dtype=int)
    for start in range(0,len(matrix),2000):
        result[start:start+2000]=cdist(matrix[start:start+2000],centres).argmin(axis=1)
    return result

def hierarchy_tree(matrix,draw,method,tag,cache=True):
    bytes_needed=len(draw)*(len(draw)-1)//2*8
    if bytes_needed>AGG_MAX_DISTANCE_BYTES:
        raise MemoryError(f'Condensed distance estimate {bytes_needed:,} exceeds declared guard.')
    # The guard concerns the distance array; total process memory is larger.
    key=hashlib.sha256((FINGERPRINT+tag+method).encode()+draw.tobytes()+matrix[draw].tobytes()).hexdigest()
    path=OUTPUT/'trees'/f'{key}.joblib'
    if cache and path.exists(): return joblib.load(path)
    tree=linkage(matrix[draw],method=method,metric='euclidean')
    if cache: joblib.dump(tree,path)
    return tree

def density_fit(training_matrix,training_weights,full_matrix,eps,min_samples,population_N):
    unique,inverse=np.unique(training_matrix,axis=0,return_inverse=True)
    mass=np.bincount(inverse,weights=training_weights*population_N)
    neighbors=NearestNeighbors(radius=eps,n_jobs=1).fit(unique)
    chunks=[];edges=0
    for start in range(0,len(unique),DBSCAN_CHUNK_SIZE):
        graph=neighbors.radius_neighbors_graph(unique[start:start+DBSCAN_CHUNK_SIZE],mode='distance')
        edges+=graph.nnz
        if edges>DBSCAN_MAX_EDGES: raise MemoryError('DBSCAN radius graph exceeded edge guard.')
        chunks.append(graph)
    graph=vstack(chunks,format='csr');graph.setdiag(0)
    model=DBSCAN(eps=eps,min_samples=min_samples,metric='precomputed',n_jobs=1)
    with warnings.catch_warnings(record=True) as caught:
        native=model.fit_predict(graph,sample_weight=mass)
    cores=model.core_sample_indices_
    predicted=np.full(len(full_matrix),-1,dtype=int)
    if len(cores):
        nearest=NearestNeighbors(n_neighbors=1,n_jobs=1).fit(unique[cores])
        for start in range(0,len(full_matrix),2000):
            distance,index=nearest.kneighbors(full_matrix[start:start+2000])
            assigned=distance[:,0]<=eps
            block=predicted[start:start+2000]
            block[assigned]=native[cores[index[assigned,0]]]
    return predicted,native[inverse],{'core_patterns':len(cores),'distinct_patterns':len(unique),
            'radius_edges':edges,'warnings':' | '.join(str(w.message) for w in caught)}


## 6. A single readable fitting function and safe caches

**What and why:** fit the chosen algorithm and return full-cohort labels plus diagnostics. GMM predicts its own components. Hierarchy uses the declared centroid extension. DBSCAN uses the nearest core pattern within the frozen radius for omitted respondents, preserving native training labels for included respondents; this extension is deterministic but is not a native DBSCAN prediction method. Border assignments may depend on the nearest-core rule, particularly near two components.

Cache fingerprints include data and scientific settings. Failed fits are returned with a status and error message; they never become fabricated labels. Non-converged GMM fits are failures for screening. All assigned clusters must exist, and full-cohort output length is checked.

In [ ]:
def fit_candidate(spec,training,seed,tag,cache=True):
    key=hashlib.sha256((FINGERPRINT+json.dumps(spec,sort_keys=True)+tag+str(seed)).encode()
                       +training.WPID_RANDOM.to_numpy().tobytes()).hexdigest()
    path=OUTPUT/'fits'/f'{key}.joblib'
    if cache and path.exists(): return joblib.load(path)
    result={'fingerprint':FINGERPRINT,'spec':spec,'status':'failed','labels':None,'diagnostics':{}}
    try:
        top=ENCODINGS[spec['encoding']];alpha=BALANCES[spec['balance']]
        full=transform(data,alpha,top);train_matrix=transform(training,alpha,top)
        with threadpool_limits(limits=1):
            if spec['algorithm']=='gmm':
                per_country=max(1,round(len(training)/training.COUNTRY_ISO3.nunique()))
                draw=country_draw(training,per_country,seed)
                # draw contains global data indices, not positions in train_matrix.
                model=GaussianMixture(n_components=spec['k'],covariance_type=spec['covariance'],
                    n_init=GMM_N_INIT,max_iter=GMM_MAX_ITER,reg_covar=GMM_REG_COVAR,random_state=seed)
                with warnings.catch_warnings(record=True) as caught: model.fit(full[draw])
                posterior=model.predict_proba(full);labels=posterior.argmax(axis=1)
                variances=model.covariances_ if spec['covariance']=='diag' else np.array([np.linalg.eigvalsh(c) for c in model.covariances_])
                result['diagnostics']={'converged':bool(model.converged_),'iterations':model.n_iter_,
                    'draw_rows':len(draw),'draw_unique_respondents':len(np.unique(draw)),
                    'ambiguous_weight_below_0_8':float(weights[posterior.max(axis=1)<.8].sum()),
                    'covariance_floor_fraction':float((variances<=1.05*GMM_REG_COVAR).mean()),
                    'warnings':' | '.join(str(w.message) for w in caught)}
                if not model.converged_: raise RuntimeError('GMM did not converge.')
            elif spec['algorithm']=='agglomerative':
                draw=country_draw(training,AGG_PER_COUNTRY,seed)
                tree=hierarchy_tree(full,draw,spec['linkage'],tag,cache)
                native=cut_tree(tree,n_clusters=[spec['k']]).ravel()
                centres=np.array([full[draw][native==j].mean(axis=0) for j in range(spec['k'])])
                labels=predict_centroids(full,centres)
                result['diagnostics']={'draw_rows':len(draw),'draw_unique_respondents':len(np.unique(draw)),
                    'native_extension_agreement':float((labels[draw]==native).mean()),
                    'native_sizes':np.bincount(native).tolist()}
            else:
                labels,native,diagnostics=density_fit(train_matrix,country_weights(training),full,
                                                    spec['eps'],spec['min_samples'],len(data))
                labels[training.index.to_numpy()]=native
                result['diagnostics']=diagnostics
        assert len(labels)==len(data)
        result.update(status='ok',labels=labels.astype('int32'))
    except (MemoryError,ValueError,RuntimeError) as error:
        result['error']=str(error)
    if cache: joblib.dump(result,path)
    return result

def label_metrics(labels):
    assigned=labels>=0
    groups=np.unique(labels[assigned])
    shares=np.array([weights[labels==j].sum() for j in groups])
    coverage=weights[assigned].sum()
    largest=shares.max()/coverage if len(shares) else np.nan
    return {'observed_k':len(groups),'coverage':coverage,'largest_assigned_share':largest,
            'dominance':len(groups)*largest,'smallest_assigned_share':shares.min()/coverage if len(shares) else np.nan,
            'sizes_assigned_percent':', '.join(f'{v/coverage*100:.1f}' for v in sorted(shares,reverse=True))}


## 7. Calibrate DBSCAN radii and declare candidates

**What and why:** k is fixed for GMM and hierarchy, but emerges from eps and min_samples for DBSCAN. For each representation, use distances to the min_samples-th neighbor on a weighted draw of reference query respondents and four distance quantiles. The training-neighbor search itself uses raw respondents, so this radius heuristic is not an exact weighted density quantile. Actual DBSCAN core formation does use aggregated analysis mass. Frozen radii allow identical hyperparameters across repetitions.

Duplicate questionnaire patterns can give zero neighbor distances. Zero-radius proposals are logged and skipped because eps must be positive; repeated positive proposals are deduplicated. This is a limitation of the radius grid, not evidence that DBSCAN has no possible solution.

In [ ]:
specs=[];radius_rows=[]
for encoding in ENCODINGS:
    for balance in BALANCES:
        for k in K_VALUES:
            for covariance in GMM_COVARIANCES:
                specs.append({'algorithm':'gmm','encoding':encoding,'balance':balance,'k':k,'covariance':covariance})
            for method in ['ward','average']:
                specs.append({'algorithm':'agglomerative','encoding':encoding,'balance':balance,'k':k,'linkage':method})
if RUN_REFERENCE:
    rng=np.random.default_rng(SEED)
    probes=rng.choice(len(data),min(DBSCAN_PROBES,len(data)),replace=True,p=weights)
    for encoding,top in ENCODINGS.items():
        for balance,alpha in BALANCES.items():
            matrix=transform(data,alpha,top)
            neighbors=NearestNeighbors(n_neighbors=max(DBSCAN_MIN_SAMPLES),n_jobs=1).fit(matrix)
            distance,_=neighbors.kneighbors(matrix[probes])
            for m in DBSCAN_MIN_SAMPLES:
                radii=np.quantile(distance[:,m-1],DBSCAN_RADIUS_QUANTILES)
                for q,eps in zip(DBSCAN_RADIUS_QUANTILES,radii):
                    radius_rows.append({'encoding':encoding,'balance':balance,'min_samples':m,'quantile':q,
                                        'eps':eps,'status':'positive' if eps>0 else 'zero_skipped'})
                for eps in sorted(set(float(x) for x in radii if x>0)):
                    specs.append({'algorithm':'dbscan','encoding':encoding,'balance':balance,'eps':eps,'min_samples':m})
    pd.DataFrame(radius_rows).to_csv(OUTPUT/'dbscan_radius_grid.csv',index=False)
    joblib.dump({'fingerprint':FINGERPRINT,'specs':specs},OUTPUT/'specifications.joblib')
elif (OUTPUT/'specifications.joblib').exists():
    saved=joblib.load(OUTPUT/'specifications.joblib')
    assert saved['fingerprint']==FINGERPRINT, 'Candidate design changed; enable reference search to rebuild.'
    specs=saved['specs']
print('Declared candidates:',len(specs),'DBSCAN calibration enabled:',RUN_REFERENCE)


## 8. Smoke test all algorithm paths on a small diagnostic dataset

**What and why:** exercise each fitter on a small, country-stratified complete-response subset, including a deliberately small DBSCAN radius. Temporarily limit hierarchy draws and GMM starts for this engineering test, then restore them even if a check fails. Do not save these fits in scientific caches or report their cluster meanings. The full data variable is restored afterwards.

Also check that weighted pattern compression yields the same DBSCAN partition as a direct weighted fit on this diagnostic sample, and compare the native hierarchy labels with the centroid extension. These tests validate implementation, not search quality.

In [ ]:
saved_data=data;saved_weights=weights
saved_agg=AGG_PER_COUNTRY;saved_starts=GMM_N_INIT
try:
    chosen=sorted(data.COUNTRY_ISO3.unique())[:5]
    parts=[data.loc[data.COUNTRY_ISO3.eq(c)].sample(n=80,random_state=SEED) for c in chosen]
    data=pd.concat(parts).reset_index(drop=True);weights=country_weights(data)
    AGG_PER_COUNTRY=40;GMM_N_INIT=1
    smoke_specs=[{'algorithm':'gmm','encoding':'013','balance':'balanced','k':3,'covariance':c} for c in GMM_COVARIANCES]
    smoke_specs += [{'algorithm':'agglomerative','encoding':'013','balance':'balanced','k':3,'linkage':m} for m in ['ward','average']]
    smoke_specs += [{'algorithm':'dbscan','encoding':'013','balance':'balanced','eps':.12,'min_samples':5}]
    smoke_rows=[]
    for spec in smoke_specs:
        result=fit_candidate(spec,data,SEED,'engineering_test',cache=False)
        assert result['status']=='ok',result.get('error')
        smoke_rows.append({**spec,**label_metrics(result['labels'])})
    matrix=transform(data,.5,3.)
    direct=DBSCAN(eps=.12,min_samples=5).fit_predict(matrix,sample_weight=weights*len(data))
    compressed,native,_=density_fit(matrix,weights,matrix,.12,5,len(data))
    assert np.array_equal(direct==-1,native==-1)
    assert adjusted_rand_score(direct,native)>.999999
    show('Engineering smoke tests only not scientific results',pd.DataFrame(smoke_rows))
finally:
    data=saved_data;weights=saved_weights
    AGG_PER_COUNTRY=saved_agg;GMM_N_INIT=saved_starts


## 9. Full reference fits and structural screening

**What and why:** fit all declared reference candidates and report failures, convergence, ambiguity, extension agreement and density coverage. Apply k≥3, largest assigned-cluster share≤40% and dominance<3. For DBSCAN require at least 90% reference weight assigned, so dropping large parts of the cohort cannot create attractive cluster sizes. This additional coverage guard is provisional and explicitly separate from the K-means rules.

Do not silently exclude technical failures. Keep their rows and reasons. Hierarchical native-versus-extension agreement is a diagnostic rather than an automatic scientific pass. Gaussian covariance-floor warnings must be inspected before interpreting a qualifying GMM.

In [ ]:
references={};reference_rows=[]
if RUN_REFERENCE:
    for number,spec in enumerate(specs):
        result=fit_candidate(spec,data,20231006,'reference')
        references[number]=result
        row={'candidate':number,**spec,'status':result['status'],**result['diagnostics']}
        if result['status']=='ok': row.update(label_metrics(result['labels']))
        else: row['error']=result.get('error','Unknown failure')
        reference_rows.append(row)
        print(f'Reference {number+1}/{len(specs)}: {spec["algorithm"]} {result["status"]}',flush=True)
    reference_table=pd.DataFrame(reference_rows)
    reference_table['structural_pass']=False
    good=reference_table.status.eq('ok')
    reference_table.loc[good,'structural_pass']=(reference_table.loc[good,'observed_k'].between(3,10)
        &reference_table.loc[good,'largest_assigned_share'].le(.4)&reference_table.loc[good,'dominance'].lt(3)
        &reference_table.loc[good,'coverage'].ge(.90))
    reference_table.to_csv(OUTPUT/'reference_screen.csv',index=False)
    joblib.dump({'fingerprint':FINGERPRINT,'results':references},OUTPUT/'references.joblib')
elif (OUTPUT/'references.joblib').exists():
    saved=joblib.load(OUTPUT/'references.joblib')
    assert saved['fingerprint']==FINGERPRINT, 'Reference design changed; rerun reference stage.'
    references=saved['results']
    reference_table=pd.read_csv(OUTPUT/'reference_screen.csv')
else:
    reference_table=pd.DataFrame()
show('Reference fits and structural screening',reference_table)


## 10. Full repetitions for structurally eligible references

**What and why:** give every structurally eligible candidate exactly 50 respondent and 50 country repetitions, with the verified v3 sample plans. Cache every fit separately. No model qualifies using only a small number of repetitions. Reference screening narrows expensive work but does not establish stability.

GMM and hierarchy also redraw their weighted fitting sample inside each selected training population. Hierarchical trees are cached once per representation, linkage and sample and cut at every requested k. DBSCAN freezes radius and density-mass scale; its group count may change. These extra algorithm-specific operations are part of the evaluated pipeline and must not be concealed when comparing with K-means.

In [ ]:
repetitions={}
if RUN_STABILITY:
    assert len(reference_table), 'Run the full reference stage first.'
    retained=reference_table.loc[reference_table.structural_pass,'candidate'].astype(int).tolist()
    for number in retained:
        spec=specs[number]
        for kind,indices in plans.items():
            results=[]
            for repeat,idx in enumerate(indices):
                result=fit_candidate(spec,data.iloc[idx],SEED+1000+repeat,f'{kind}_{repeat:02d}')
                results.append(result)
                print(f'Candidate {number} {kind} repetition {repeat+1}/{REPETITIONS}: {result["status"]}',flush=True)
            repetitions[(number,kind)]=results
    joblib.dump({'fingerprint':FINGERPRINT,'results':repetitions},OUTPUT/'repetitions.joblib')
elif (OUTPUT/'repetitions.joblib').exists():
    saved=joblib.load(OUTPUT/'repetitions.joblib')
    assert saved['fingerprint']==FINGERPRINT, 'Repetition design changed; rerun stability stage.'
    repetitions=saved['results']
print('Candidate and test combinations with repetition outputs:',len(repetitions))


## 11. ARI summaries with explicit noise and failed fit handling

**What and why:** calculate all 1,225 pairwise ARIs and fifty reference comparisons per candidate/test. For non-density algorithms all respondents are assigned. For DBSCAN, the primary ARI uses respondents assigned in both fits; the all-label ARI (including noise as a label) is secondary. Report their jointly assigned weight, reference coverage, k distributions and noise membership stability. Shared-country comparisons are additional diagnostics, not substitutes for the common population.

Qualifying requires both medians and both lower deciles≥0.8, the existing reference size and dominance criteria, all fifty fits successful, and no repetition with fewer than three or more than ten groups. For DBSCAN also require the tenth percentile of jointly assigned weight≥90% in both tests. These coverage conditions prevent an easy noise-versus-everyone split or a tiny consistently assigned subset from passing. Quantiles use dependent comparisons and are not confidence intervals. Failed repetitions cannot be dropped to improve the score.

In [ ]:
# Load the completed, saved pairwise comparisons; do not recompute them.
stability = pd.read_csv(OUTPUT / 'stability_summary.csv')
pair_table = pd.read_csv(OUTPUT / 'pairwise_ari.csv')
winners = [int(number) for number, group in stability.groupby('candidate')
           if set(group.experiment) == set(plans) and group.stability_pass.all()]
show('Completed stability tests: median and lower-decile ARI', stability)
print('Candidates passing both complete tests:', winners)


## 12. Original response profiles and comparison with B and E

**What and why:** characterize every successful reference, not just winners. Report absolute cohort shares and shares conditional on assignment, original worry categories, overall worry distributions and both harm types. Noise is a separate description row, never a substantive profile. Compare saved B/E assignments on the same respondents and calculate directional refinement purity, not just ARI.

For density solutions report coverage alongside every comparison. High refinement on an assigned subset cannot establish refinement of the full population. Profile labels must be justified by distributions; do not automatically name a group using its largest average feature. The numerical finalist list is not a scientific conclusion.

In [ ]:
baseline_dir=ROOT/'course_artifacts/week04_experiment1_complete_overall_worry_v3'
baselines={'B':joblib.load(baseline_dir/'reference_012_balanced_k4.joblib')['labels'],
           'E':joblib.load(baseline_dir/'reference_013_worry_emphasized_k8.joblib')['labels']}
assert all(len(labels)==len(data) for labels in baselines.values())
observed=data[FEATURES].to_numpy(float);overall=data[WORRY].mean(axis=1).to_numpy()
profile_rows=[];category_rows=[];comparison_rows=[]
for number,result in references.items():
    if result['status']!='ok': continue
    labels=result['labels'];coverage=weights[labels>=0].sum()
    for cluster in np.unique(labels):
        inside=labels==cluster;share=weights[inside].sum()
        order=np.argsort(overall[inside]);v=overall[inside][order];w=weights[inside][order]
        q=np.interp([.1,.5,.9],np.cumsum(w)/w.sum(),v)
        row={'candidate':number,'cluster':cluster,'is_noise':cluster==-1,'cohort_share':share,
             'assigned_share':share/coverage if cluster>=0 else np.nan,
             'overall_worry_mean':np.average(overall[inside],weights=weights[inside]),
             'overall_worry_p10':q[0],'overall_worry_median':q[1],'overall_worry_p90':q[2]}
        row.update(dict(zip(FEATURES,np.average(observed[inside],weights=weights[inside],axis=0))))
        profile_rows.append(row)
        for j,feature in enumerate(WORRY):
            for value in [0,1,2]:
                category_rows.append({'candidate':number,'cluster':cluster,'feature':feature,'response':value,
                    'proportion':weights[inside&(observed[:,j]==value)].sum()/share})
    assigned=labels>=0
    for name,baseline in baselines.items():
        a=baseline[assigned];b=labels[assigned]
        if not len(a):continue
        contingency=pd.crosstab(a,b,values=weights[assigned],aggfunc='sum').fillna(0).to_numpy()
        comparison_rows.append({'candidate':number,'baseline':name,'coverage':coverage,
            'ARI_assigned':adjusted_rand_score(a,b),'candidate_refines_baseline':contingency.max(axis=0).sum()/coverage,
            'baseline_refines_candidate':contingency.max(axis=1).sum()/coverage})
profiles=pd.DataFrame(profile_rows);categories=pd.DataFrame(category_rows);comparisons=pd.DataFrame(comparison_rows)
for name,table in [('original_response_profiles',profiles),('worry_categories',categories),('B_E_comparisons',comparisons)]:
    if len(table):table.to_csv(OUTPUT/f'{name}.csv',index=False)
if len(profiles):
    PLOTS=OUTPUT/'profiles';PLOTS.mkdir(exist_ok=True)
    for number in references:
        f=profiles.loc[profiles.candidate.eq(number)&~profiles.is_noise]
        if not len(f):continue
        c=categories.loc[categories.candidate.eq(number)&categories.response.eq(2)&categories.cluster.ge(0)]
        fig,axes=plt.subplots(1,3,figsize=(16,5))
        for ax,columns,title in zip(axes,[WORRY,PERSONAL,KNOWN],['Very worried percent','Personal harm percent','Known person harm percent']):
            values=c.pivot(index='cluster',columns='feature',values='proportion') if columns==WORRY else f.set_index('cluster')[columns]
            sns.heatmap(values.reindex(columns=columns)*100,annot=True,fmt='.0f',vmin=0,vmax=100,cmap='Blues',cbar=False,ax=ax)
            ax.set(title=title,xlabel='Risk domain',ylabel='Reference cluster')
            ax.set_xticklabels([d.replace('_',' ') for d in (DOMAINS[:6] if columns==WORRY else DOMAINS)],rotation=40,ha='right')
        fig.suptitle(f'Candidate {number} | {references[number]["spec"]}');fig.tight_layout()
        fig.savefig(PLOTS/f'candidate_{number}.png',dpi=140);plt.close(fig)
show('Numerically qualifying original response profiles',profiles.loc[profiles.candidate.isin(winners)] if len(profiles) else profiles)
show('Agreement and directional refinement with B and E',comparisons)


## 13. Compare all qualifying solutions for approximate refinement

**What and why:** compare every pair among the new numerically qualifying candidates and saved B/E, not just a selected favorable pair. Orient different-k pairs from fewer to more groups. For equal k report optimal one-to-one weighted agreement instead. Original labels remain available in overlap tables.

For DBSCAN comparisons, restrict to jointly assigned respondents and report their weighted coverage. Conditional refinement cannot establish nesting for the omitted population. Flag strong approximate relationships using overall purity≥90% and every finer-cluster purity≥80%, plus joint coverage≥90%. These illustrative graph rules are separate from the model-selection filters. Approximate nesting is not necessarily transitive, so do not infer an untested lattice. A finer model is preferred only when its new distinction is scientifically useful and it meets the stability criteria.

In [ ]:
retained_labels=dict(baselines)
for number in winners: retained_labels[f'New {number}']=references[number]['labels']
refinement_rows=[];overlap_rows=[]
for first,second in itertools.combinations(retained_labels,2):
    a,b=retained_labels[first],retained_labels[second]
    ka=len(np.unique(a[a>=0]));kb=len(np.unique(b[b>=0]))
    if ka>kb: first,second=second,first;a,b=b,a;ka,kb=kb,ka
    joint=(a>=0)&(b>=0);coverage=weights[joint].sum()
    table=pd.crosstab(a[joint],b[joint],values=weights[joint],aggfunc='sum').fillna(0)
    if table.empty:continue
    counts=table.to_numpy();score=counts.max(axis=0).sum()/coverage
    worst=(counts.max(axis=0)/counts.sum(axis=0)).min()
    r,c=linear_sum_assignment(-counts)
    agreement=counts[r,c].sum()/coverage if ka==kb else np.nan
    refinement_rows.append({'coarse':first,'fine':second,'coarse_k':ka,'fine_k':kb,
        'joint_coverage':coverage,'refinement_purity':score,'worst_fine_purity':worst,
        'one_to_one_agreement':agreement,'strong_approximate_refinement':bool(ka<kb and coverage>=.90 and score>=.90 and worst>=.80)})
    for i in table.index:
        for j in table.columns:
            overlap_rows.append({'coarse':first,'fine':second,'coarse_cluster':i,'fine_cluster':j,
                'joint_weight':table.loc[i,j],'fine_cluster_purity':table.loc[i,j]/table[j].sum()})
refinement=pd.DataFrame(refinement_rows)
refinement.to_csv(OUTPUT/'retained_pairwise_refinement.csv',index=False)
pd.DataFrame(overlap_rows).to_csv(OUTPUT/'retained_membership_overlaps.csv',index=False)
show('Pairwise approximate refinement and same k correspondence',refinement)
if not winners:
    print('No new numerically qualified model yet: this table contains the B/E baseline comparison only.')


## Conclusions to write after the complete experiment

Do not declare a new winner until both 50-repetition tests are complete. Start with the numerical filters, then inspect semantic coherence, noise coverage, GMM Gaussian misspecification and hierarchy extension diagnostics. A model can pass stability yet merely reproduce a single binary answer. A model can fail transport while its native training hierarchy remains meaningful. Distinguish these conclusions.

Compare qualifying alternatives with B and E using original profiles and membership overlaps. Approximate refinement may justify keeping a finer solution only if the additional distinction is useful. If no alternative passes, report that outcome without loosening thresholds silently. If changing fitting sample size, regularization or DBSCAN coverage constraints is scientifically justified, make a new version and document it before rerunning. No algorithm has been evaluated by this notebook until the execution switches are enabled.

### Implementation references

These explain API capabilities, not the scientific adequacy of the design: [GaussianMixture](https://scikit-learn.org/stable/modules/generated/sklearn.mixture.GaussianMixture.html), [agglomerative clustering](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.AgglomerativeClustering.html), [DBSCAN and its sample weights](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html). Installed package versions should be recorded alongside results.


## Conclusions from the completed search

All 288 reference specifications were screened. Of these, 60 passed the structural screen and received 50 respondent and 50 stratified-country repetitions: 6,000 successful fits. The 23 DBSCAN reference failures remain recorded in the reference-screen table; they are computational failures, not evidence against all possible DBSCAN settings.

Only candidate 170, **Ward agglomerative clustering, 0–1–3 worry encoding, worry-emphasized balance, k=8**, passes every agreed filter. Respondent median/lower-decile ARI is 0.943/0.888; country median/lower-decile ARI is 0.929/0.860. Its largest cluster is 34.8%, dominance D=2.78, and assigned coverage is 100%. Ward k=7 is a near-miss: country lower-decile ARI 0.784; it is not retained under the unchanged 0.80 rule.

### Interpreting the eight profiles

Labels are descriptive summaries, not diagnoses or proof of distinct population types. In cluster-ID order: low worry (34.8%), mental-health-focused (11.1%), food-focused (9.4%), severe-weather-focused (7.3%), crime-focused (12.6%), water-focused (8.8%), broadly elevated worry (10.2%), and traffic-focused (5.8%). These descriptions use original 0–1–2 worry responses, not the transformed clustering coordinates. The thematic groups frequently report very high worry in their named domain; harm is more heterogeneous and does not define a separate harm-heavy profile. For example, personal severe-weather harm is 21.7% in the weather group and personal traffic harm is 16.9% in the traffic group. The broadly elevated group has greater harm across several domains, but worry and harm are not interchangeable.

### Comparison with the K-means finalists

Ward and K-means E both have eight clusters: ARI=0.820 and optimal matched membership=82.3%. They corroborate a thematic worry structure but reorganize some boundaries: Ward distinguishes food whereas E distinguishes crime/traffic configurations more finely. Neither is a strong approximate refinement of the other. Ward versus B has ARI=0.245 and refinement purity toward B=0.613: Ward does not simply subdivide B. Keep B as the balanced worry/harm perspective and E/Ward as alternative thematic perspectives; do not automatically replace one by the other.

The approximate-refinement graph has three nodes (B, E, Ward) and **no strong refinement edges** under purity ≥0.90 and worst-fine-cluster purity ≥0.80. It is not a mathematical lattice. GMM and average linkage produced no retained candidate in this search; DBSCAN produced no structurally eligible candidate. This conclusion concerns the tested pipelines, including weighted bootstrap fitting and centroid-based extension for agglomerative clustering, not intrinsic superiority of K-means over every alternative.

The original terminal run completed fitting and stability calculations, then encountered a plotting-backend error. Reporting was recovered using a noninteractive backend and saved caches, without refitting or repeating pairwise ARI calculations. This version loads those saved stability tables deliberately. Held-out countries have not been used to select the solution. Regional analysis remains a separate future experiment.


### Saved result visuals

The profile chart below uses original responses. The graph records the absence of strong directed refinement relations; similarity (ARI) is a separate comparison.

In [ ]:
# Saved completed-result figure: candidate_170.png

In [ ]:
# Saved completed-result figure: approximate_refinement_graph.png